[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day33-fp8-and-fp4-quantization.ipynb)

# Day 33 — FP8 (and FP4) Quantization

**Tags:** `CPU-OK` · **Time:** 30–60 min · **Prereqs:** Day 29 (affine quantization), Day 31 (AWQ salient channels), Day 32 (fp8 KV cache)

FP8 is quantization that barely deserves the name: one per-block scale plus the
format's own exponent/mantissa grid. Today you simulate E4M3 on CPU/T4 (cast
weights through `torch.float8_e4m3fn`, compute stays fp16), measure the
perplexity cost on GPT-2, do the Hopper throughput math, and demo why FP4
needs per-32-element microscaling. Native FP8 compute (sm_90/H100) is READ-ONLY.



In [ ]:
# Cell 1: one-time installs. torch/numpy ship with Colab; transformers + datasets
# are needed for the GPT-2 perplexity side-quest (Part 3). CPU-only day.
%pip install --quiet transformers datasets
import torch, math
print("torch", torch.__version__, "| device: cpu (all cells run on CPU)")
print("float8_e4m3fn available:", hasattr(torch, "float8_e4m3fn"))
# Expected: torch 2.x.x | device: cpu (all cells run on CPU)
#           float8_e4m3fn available: True



## Part 1 — E4M3 representable range: which magnitudes survive?

E4M3 = 1 sign + 4 exponent + 3 mantissa bits. Max finite value 448, smallest
subnormal ~0.00195. Round-trip four probe magnitudes through the dtype and
read off what the format does to each.



In [ ]:
# Cell 2: fp8_math — E4M3 representability of [1e-4, 0.1, 100, 1000]
probes = torch.tensor([1e-4, 0.1, 100.0, 1000.0])
back = probes.to(torch.float8_e4m3fn).to(torch.float32)
for v, b in zip(probes.tolist(), back.tolist()):
    err = abs(b - v) / v * 100 if v else 0.0
    status = "OK" if b == v or err < 5 else ("UNDERFLOW -> 0" if b == 0 else "SATURATED")
    print(f"{v:>8} -> {b:<12} rel err {err:5.2f}%  {status}")
# Expected:
#    0.0001 -> 0.0          rel err 100.00%  UNDERFLOW -> 0
#       0.1 -> 0.1015625    rel err  1.56%   OK
#     100.0 -> 96.0         rel err  4.00%   OK
#    1000.0 -> 448.0        rel err 55.20%   SATURATED  (E4M3 max)



## Part 2 — Simulated-FP8 linear layer: the 5-line quantization

The whole FP8 "algorithm": per-row scale to the E4M3 range, round to the grid,
cast back. Compute stays fp16 — exactly what the T4/H100 simulation path does
in production (real FP8 *compute* needs sm_90).



In [ ]:
# Cell 3: simulated FP8 weight quantization + fp16 matmul error
E4M3_MAX = 448.0
torch.manual_seed(33)
W = torch.randn(512, 512, dtype=torch.float16) * 0.6   # stand-in weight matrix
X = torch.randn(64, 512, dtype=torch.float16)          # stand-in activations

def quantize_fp8_sim(w):
    # per-row scale into E4M3 range, round to grid, dequantize to fp16
    amax = w.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8)
    scale = E4M3_MAX / amax
    wq = (w * scale).to(torch.float8_e4m3fn).to(torch.float16) / scale
    return wq

Wq = quantize_fp8_sim(W)
out_fp16 = X @ W.T
out_fp8  = X @ Wq.T
rel = (out_fp8 - out_fp16).abs().amax() / out_fp16.abs().amax()
print(f"max rel output err: {rel.item()*100:.2f}%")
print(f"mean rel weight err: {((Wq-W).abs()/W.abs().clamp_min(1e-8)).mean().item()*100:.2f}%")
# Expected: max rel output err ~0.3-1.5% | mean rel weight err ~2-6%
# (the format's ~3-mantissa-bit budget: visible, small, no calibration needed)



## Part 3 — Perplexity side-quest: GPT-2 fp16 vs simulated-FP8 (T4 optional, CPU fine)

Load `gpt2`, evaluate a small WikiText-2 slice twice: once in fp16, once with
every weight matrix simulated to E4M3 (per-tensor scale, cast back to fp16).
The *delta* is the number that matters.



In [ ]:
# Cell 4: GPT-2 fp16 vs simulated-FP8 perplexity on a WikiText-2 slice
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch.nn as nn

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2", torch_dtype=torch.float16).eval()
data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
text = "\n\n".join(data["text"])[:20000]
ids = tok(text, return_tensors="pt").input_ids[0]

@torch.no_grad()
def ppl(m, ids, seqlen=128, nchunks=4):
    nll, n = 0.0, 0
    for i in range(nchunks):
        chunk = ids[i*seqlen:(i+1)*seqlen].unsqueeze(0)
        loss = m(chunk, labels=chunk).loss
        nll += loss.item() * seqlen; n += seqlen
    return math.exp(nll / n)

ppl_fp16 = ppl(model, ids)
print(f"fp16 ppl: {ppl_fp16:.3f}")

# simulate FP8: per-tensor scale every weight/bias through E4M3, back to fp16
with torch.no_grad():
    for name, p in model.named_parameters():
        if p.dim() >= 2:                      # weight matrices only
            amax = p.abs().amax().clamp_min(1e-8)
            s = E4M3_MAX / amax
            p.copy_((p * s).to(torch.float8_e4m3fn).to(torch.float16) / s)
ppl_fp8 = ppl(model, ids)
print(f"fp8-sim ppl: {ppl_fp8:.3f}   (delta {ppl_fp8 - ppl_fp16:+.3f})")
# Expected: fp16 ppl ~30-31 (slice-dependent; yours will differ slightly)
#           fp8-sim delta +0.02 to +0.10 -- in the noise, no Hessian, no calibration
# On CPU this cell takes a few minutes; on a T4 it is under a minute.



## Part 4 — The Hopper throughput math (no GPU needed)

Decode is bandwidth-bound: ceiling = HBM bandwidth / bytes-per-token.
Prefill is compute-bound: latency = FLOPs / tensor-core throughput.



In [ ]:
# Cell 5: 8B fp16 vs FP8 on H100; DeepSeek-V3 MoE decode
BW = 3.35e12          # H100 HBM3, bytes/s
TF_FP16, TF_FP8 = 989e12, 1979e12   # H100 dense TFLOP/s

for name, gb in [("fp16", 16), ("fp8", 8)]:
    ceil = BW / (gb * 1e9)
    print(f"8B {name:>4}: decode ceiling {ceil:6.0f} tok/s")

flops = 2 * 8e9 * 2000              # 2*params*seq, 2k-token prefill
print(f"prefill 2k: fp16 {flops/TF_FP16*1e3:4.1f} ms | fp8 {flops/TF_FP8*1e3:4.1f} ms")

for name, active_gb in [("fp16", 74), ("fp8", 37)]:   # DS-V3: 37B active/token
    print(f"DS-V3 {name:>4}: decode ceiling {BW/(active_gb*1e9):4.0f} tok/s")
# Expected:
# 8B fp16: decode ceiling    209 tok/s
# 8B  fp8: decode ceiling    419 tok/s
# prefill 2k: fp16 32.4 ms | fp8 16.2 ms
# DS-V3 fp16: decode ceiling   45 tok/s
# DS-V3  fp8: decode ceiling   90 tok/s



## Part 5 — NVFP4 microscaling: why per-32 scales are load-bearing

E2M1 (1 sign, 2 exp, 1 mantissa) spans only +-6 with ~16 grid values. A single
per-tensor scale cannot cover a tensor's range at 1-mantissa-bit resolution --
so NVFP4 (Blackwell) gives every 32-element block its own FP8 scale. Compare
per-tensor FP4 vs microscaled FP4 on the same weight matrix.



In [ ]:
# Cell 6: E2M1 simulation, per-tensor scale vs per-32 microscale
E2M1_GRID = torch.tensor([0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
E2M1_MAX = 6.0

def quant_e2m1(w_scaled):
    # nearest grid point by sign * grid (symmetric), saturate at 6
    s = torch.sign(w_scaled)
    a = w_scaled.abs().clamp_max(E2M1_MAX)
    idx = (a.unsqueeze(-1) - E2M1_GRID).abs().argmin(-1)
    return s * E2M1_GRID[idx]

torch.manual_seed(33)
W = torch.randn(512, 512) * 0.6

# (a) naive per-tensor FP4
s_t = E2M1_MAX / W.abs().amax()
W_t = quant_e2m1(W * s_t) / s_t
# (b) microscaled: per-32-element FP8 scale along rows
Wb = W.view(512, 16, 32)
s_b = (E2M1_MAX / Wb.abs().amax(dim=-1, keepdim=True)).clamp_min(1e-8)
W_b = (quant_e2m1(Wb * s_b) / s_b).view(512, 512)

def mrel(a, b): return ((a - b).abs() / b.abs().clamp_min(1e-3)).mean().item() * 100
print(f"per-tensor FP4  mean rel err: {mrel(W_t, W):5.1f}%   (collapses)")
print(f"microscaled FP4 mean rel err: {mrel(W_b, W):5.1f}%   (usable)")
print(f"E4M3-sim      mean rel err: {mrel(quantize_fp8_sim(W.half()).float(), W):5.1f}%   (reference)")
# Expected: per-tensor FP4 ~15-40% | microscaled ~3-8% | E4M3 ~2-6%
# The scale granularity -- not the format -- is what makes 4-bit float work.



## Wrap-up

You simulated the full FP8 story on a free CPU: the format's range limits,
the 5-line quantization that needs no calibration, a real perplexity delta in
the noise, the Hopper 2x/2x math, and why FP4 demands microscaling.

**Write in your notes:** the "when FP8 vs INT4" paragraph -- name the GPU
generation that flips the decision (Hopper / sm_90) and the two quantities
each format optimizes (throughput vs capacity).

**Tomorrow:** Day 34 -- GGUF and llama.cpp K-quants: running big models on a
laptop CPU.

